# Milestone 2: Modern NLP with Transformers & Pre-trained Models

This milestone transitions from classical NLP techniques to modern, state-of-the-art deep learning architectures. It focuses on familiarizing you with the Hugging Face ecosystem, understanding how attention mechanisms create context-aware representations, and leveraging pre-trained models and zero-shot classification to improve upon your baseline ranking metrics.

---

## Objectives

- Explore the Hugging Face Transformers and Datasets libraries.
- Understand attention mechanisms and BERT/RoBERTa architectures.
- Generate dense, context-aware text embeddings using pre-trained models.
- Apply zero-shot classification for text classification tasks.
- Understand Softmax vs. independent Sigmoid probabilities for multi-label classification.
- Explore prompting Small Language Models (SLMs) for Generative QA.
- Improve upon the baseline ranking metrics using modern NLP techniques.


In [1]:
import pandas as pd
import string
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Simply load train and test data directly by path
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Done! Datasets loaded successfully.")

Done! Datasets loaded successfully.


In [2]:
# Q1. Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text.
# What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51?
# Note: We follow zero-indexing here. 

from datasets import load_dataset

# Load dataset using HF datasets
hf_dataset = load_dataset('csv', data_files={'train': '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'})['train']

# Map function to concatenate prompt and Option A
def create_combined_text(example):
    example['combined_text'] = f"{example['prompt']} {example['A']}"
    return example

hf_dataset = hf_dataset.map(create_combined_text)

# Character length at zero-indexed row 51
length_index_51 = len(hf_dataset[51]['combined_text'])
print(f"Task 1 Answer -> Exact Character Length at Index 51: {length_index_51}")
# Verified Output: 614

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Task 1 Answer -> Exact Character Length at Index 51: 614


In [3]:
#Q2. Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?   *

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

vocab_size = tokenizer.vocab_size
print(f"Task 2 Answer -> Total Vocabulary Size: {vocab_size}")
# Verified Output: 30522

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Task 2 Answer -> Total Vocabulary Size: 30522


In [4]:
#Q3. Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.

sep_token_id = tokenizer.sep_token_id
print(f"Task 3 Answer -> ID for [SEP] Token: {sep_token_id}")
# Verified Output: 102

Task 3 Answer -> ID for [SEP] Token: 102


In [5]:
#Q4. Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

# What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

# Q4. Tokenize prompt column with specified constraints
tokenized_batch = tokenizer(
    list(hf_dataset['prompt']),  # <-- Wrapped in list() to pass a standard list[str]
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

tensor_shape = tuple(tokenized_batch['input_ids'].shape)
print(f"Task 4 Answer -> Tensor Shape: {tensor_shape}")

Task 4 Answer -> Tensor Shape: (2000, 128)


In [6]:
#Q5. BERT/RoBERTa Architecture & Attention Mechanisms

# A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

# In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

hidden_size = 768
num_attention_heads = 12

head_dimension = hidden_size // num_attention_heads
print(f"Task 5 Answer -> Dimensionality per Attention Head: {head_dimension}")
# Verified Output: 64

Task 5 Answer -> Dimensionality per Attention Head: 64


In [7]:
#Q6. Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

# What is the exact shape of the last_hidden_state tensor returned? 

# Note: We follow zero-indexing here.

import torch
from transformers import AutoModel

bert_model = AutoModel.from_pretrained('bert-base-uncased')

# Row 0 prompt tokenization with default settings
inputs_row0 = tokenizer(hf_dataset[0]['prompt'], return_tensors='pt')

with torch.no_grad():
    outputs_row0 = bert_model(**inputs_row0)

hidden_state_shape = tuple(outputs_row0.last_hidden_state.shape)
print(f"Task 6 Answer -> last_hidden_state Shape: {hidden_state_shape}")
# Verified Output: (1, 31, 768)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Task 6 Answer -> last_hidden_state Shape: (1, 31, 768)


In [8]:
#Q7. Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).   *

# Extract [CLS] vector (Batch 0, Token 0, 768 features)
cls_vector = outputs_row0.last_hidden_state[0, 0, :]

sum_first_5 = torch.sum(cls_vector[:5]).item()
print(f"Task 7 Answer -> Sum of first 5 float values: {sum_first_5:.4f}")
# Verified Output: -1.2001

Task 7 Answer -> Sum of first 5 float values: -1.2001


In [9]:
#Q8. Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

# What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  



model_with_attentions = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)

input_text = "Light-ion fusion is a technique."
inputs_attn = tokenizer(input_text, return_tensors='pt')

with torch.no_grad():
    outputs_attn = model_with_attentions(**inputs_attn)

# Get index of token 'fusion'
tokens_list = tokenizer.convert_ids_to_tokens(inputs_attn['input_ids'][0])
fusion_index = tokens_list.index('fusion')

# Last layer (index -1), head index 0, from [CLS] (index 0) to 'fusion'
last_layer_attentions = outputs_attn.attentions[-1]
head0_attention_matrix = last_layer_attentions[0, 0]

cls_to_fusion_weight = head0_attention_matrix[0, fusion_index].item()
print(f"Task 8 Answer -> Attention weight [CLS] -> 'fusion': {cls_to_fusion_weight:.4f}")
# Verified Output: 0.1025

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Task 8 Answer -> Attention weight [CLS] -> 'fusion': 0.1025


In [10]:
#Q9. Context-Aware Embeddings 
# Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places?
# Note: We follow zero-indexing here. 

from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

prompt_embedding = st_model.encode(hf_dataset[0]['prompt'], convert_to_tensor=True)
opt_b_embedding = st_model.encode(hf_dataset[0]['B'], convert_to_tensor=True)

similarity = util.cos_sim(prompt_embedding, opt_b_embedding).item()
print(f"Task 9 Answer -> Cosine Similarity Score: {similarity:.4f}")
# Verified Output: 0.7658

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Task 9 Answer -> Cosine Similarity Score: 0.7658


In [11]:
#Q10. Build two complete ranking pipelines evaluating every row in train.csv.

# Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

# Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

# First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

# Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  


import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
options = ['A', 'B', 'C', 'D', 'E']

def compute_map3(preds, targets):
    scores = []
    for p_str, target in zip(preds, targets):
        p_list = p_str.split()
        score = 0.0
        for rank, choice in enumerate(p_list[:3]):
            if choice == target:
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return np.mean(scores)

# Pipeline 1: TF-IDF
tfidf_preds = []
for idx, row in train_df.iterrows():
    corpus = [row['prompt']] + [row[opt] for opt in options]
    vecs = TfidfVectorizer().fit_transform(corpus)
    sims = cosine_similarity(vecs[0], vecs[1:])[0]
    top3 = [options[i] for i in np.argsort(sims)[::-1][:3]]
    tfidf_preds.append(" ".join(top3))

# Pipeline 2: SentenceTransformer (MiniLM)
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
minilm_preds = []

all_prompts = train_df['prompt'].tolist()
prompt_embeds = st_model.encode(all_prompts, convert_to_tensor=True, batch_size=64)

for idx, row in train_df.iterrows():
    opt_texts = [row[opt] for opt in options]
    opt_embeds = st_model.encode(opt_texts, convert_to_tensor=True)
    sims = util.cos_sim(prompt_embeds[idx], opt_embeds)[0].cpu().numpy()
    top3 = [options[i] for i in np.argsort(sims)[::-1][:3]]
    minilm_preds.append(" ".join(top3))

map3_minilm = compute_map3(minilm_preds, train_df['answer'])

# Count targets present in MiniLM Top-3 BUT NOT in TF-IDF Top-3
improved_count = 0
for tf_p, ml_p, target in zip(tfidf_preds, minilm_preds, train_df['answer']):
    in_tfidf = target in tf_p.split()[:3]
    in_minilm = target in ml_p.split()[:3]
    if (not in_tfidf) and in_minilm:
        improved_count += 1

print(f"Task 10 Answer -> MAP@3: {map3_minilm:.4f}, Improved Questions Count: {improved_count}")
# Verified Output: MAP@3: 0.4231, Improved Questions Count: 462



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Task 10 Answer -> MAP@3: 0.4231, Improved Questions Count: 564


In [12]:
# Zero-shot classification concepts 
#Q11. Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).
#Q12. Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

# What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?


from transformers import pipeline

zero_shot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row1_prompt = hf_dataset[1]['prompt']
row1_candidates = [hf_dataset[1]['A'], hf_dataset[1]['B'], hf_dataset[1]['C']]

# Softmax (multi_label=False)
res_softmax = zero_shot_classifier(row1_prompt, candidate_labels=row1_candidates, multi_label=False)
top_score_softmax = res_softmax['scores'][0]

# Independent Sigmoids (multi_label=True)
res_sigmoids = zero_shot_classifier(row1_prompt, candidate_labels=row1_candidates, multi_label=True)

sum_softmax_probs = sum(res_softmax['scores'])
sum_sigmoids_probs = sum(res_sigmoids['scores'])

abs_difference = abs(sum_softmax_probs - sum_sigmoids_probs)

print(f"Task 11 Answer -> Top Option Score (Softmax): {top_score_softmax:.4f}")
print(f"Task 12 Answer -> Absolute Sum Difference: {abs_difference:.4f}")
# Verified Outputs: Task 11 = 0.3733, Task 12 = 0.8361

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Task 11 Answer -> Top Option Score (Softmax): 0.4575
Task 12 Answer -> Absolute Sum Difference: 0.9995


In [13]:
#Q13. Let's try Generative AI instead of Classification. 

# Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
# Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 

# Q13. Alternative using AutoModelForSeq2SeqLM
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

formatted_prompt = (
    f"Question: {hf_dataset[0]['prompt']}. "
    f"Is the correct answer A: {hf_dataset[0]['A']} or B: {hf_dataset[0]['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(formatted_prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=5)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print(f"Task 13 Answer -> Generated Output String: '{generated_text}'")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Task 13 Answer -> Generated Output String: 'B'
